In [0]:
# ============================================================
# NOTEBOOK 3 — GOLD LAYER (Business KPIs & Aggregations)
# Reads from : brazilian-ecommerce.silver.orders_master
# Writes to  : brazilian-ecommerce.gold.*
# ============================================================

CATALOG       = "brazilian-ecommerce"
SILVER_SCHEMA = "silver"
GOLD_SCHEMA   = "gold"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG}`.`{GOLD_SCHEMA}`")

SILVER_TABLE = f"`{CATALOG}`.`{SILVER_SCHEMA}`.`orders_master`"

print(f"Silver source : {SILVER_TABLE}")
print(f"Gold target   : {CATALOG}.{GOLD_SCHEMA}")

In [0]:
from pyspark.sql.functions import (
    col, year, month, quarter, date_format,
    sum as spark_sum, avg, count, countDistinct,
    round as spark_round, when, lit, max as spark_max,
    min as spark_min, datediff, first
)
from pyspark.sql.window import Window

# Read Silver master table
df_silver = spark.table(SILVER_TABLE)

# Add time dimension columns — essential for all time-series KPIs
df_silver = (
    df_silver
    .withColumn("order_year",    year(col("order_purchase_timestamp")))
    .withColumn("order_month",   month(col("order_purchase_timestamp")))
    .withColumn("order_quarter", quarter(col("order_purchase_timestamp")))
    .withColumn("order_month_str",
                date_format(col("order_purchase_timestamp"), "yyyy-MM"))
)

total = df_silver.count()
print(f"Silver rows loaded : {total:,}")
df_silver.select(
    "order_year", "order_month", "order_quarter", "order_month_str"
).distinct().orderBy("order_month_str").show(10)

In [0]:
# Most fundamental business KPI — revenue trend over time per category
# This answers: "Which categories are growing? Which are declining?"

df_gold_revenue = (
    df_silver
    .filter(col("order_status") == "DELIVERED")   # only count completed orders
    .groupBy(
        "order_year",
        "order_month",
        "order_month_str",
        "order_quarter",
        "product_category_name"
    )
    .agg(
        spark_round(spark_sum("line_total"),    2).alias("total_revenue"),
        spark_round(spark_sum("freight_value"), 2).alias("total_freight"),
        spark_round(spark_sum("price"),         2).alias("total_product_revenue"),
        spark_round(avg("price"),               2).alias("avg_order_value"),
        countDistinct("order_id")                 .alias("total_orders"),
        countDistinct("customer_unique_id")       .alias("unique_customers"),
        spark_round(avg("actual_delivery_days"),2) .alias("avg_delivery_days"),
        spark_sum(when(col("is_late_delivery") == True, 1).otherwise(0))
                                                  .alias("late_order_count")
    )
    .withColumn("late_delivery_rate",
                spark_round(
                    col("late_order_count") / col("total_orders") * 100, 2
                ))
    .withColumn("freight_as_pct_revenue",
                spark_round(
                    col("total_freight") / col("total_revenue") * 100, 2
                ))
    .orderBy("order_year", "order_month", col("total_revenue").desc())
)

print(f"Revenue KPI rows : {df_gold_revenue.count():,}")
display(df_gold_revenue.limit(20))

In [0]:
# CLV = total revenue a customer has generated across all orders
# One of the most important metrics for e-commerce businesses

df_gold_clv = (
    df_silver
    .filter(col("order_status") == "DELIVERED")
    .groupBy("customer_unique_id", "customer_state")
    .agg(
        # Financial metrics
        spark_round(spark_sum("line_total"),    2).alias("lifetime_revenue"),
        spark_round(avg("line_total"),          2).alias("avg_order_value"),
        spark_round(spark_sum("freight_value"), 2).alias("total_freight_paid"),

        # Behavioural metrics
        countDistinct("order_id")               .alias("total_orders"),
        countDistinct("product_category_name")  .alias("unique_categories_bought"),

        # Time metrics
        spark_min("order_purchase_timestamp")   .alias("first_order_date"),
        spark_max("order_purchase_timestamp")   .alias("last_order_date"),

        # Quality metrics
        spark_sum(when(col("is_late_delivery") == True, 1)
                  .otherwise(0))                .alias("late_deliveries_received")
    )
    # Customer tenure in days
    .withColumn("customer_tenure_days",
                datediff(
                    col("last_order_date"),
                    col("first_order_date")
                ))
    # Customer segment based on lifetime revenue
    .withColumn("customer_segment",
                when(col("lifetime_revenue") >= 1000, lit("high_value"))
                .when(col("lifetime_revenue") >= 300,  lit("mid_value"))
                .otherwise(lit("low_value")))
)

print(f"CLV table rows : {df_gold_clv.count():,}")

# Segment distribution
display(
    df_gold_clv.groupBy("customer_segment")
    .agg(
        count("*").alias("customer_count"),
        spark_round(avg("lifetime_revenue"), 2).alias("avg_clv"),
        spark_round(spark_sum("lifetime_revenue"), 2).alias("total_revenue")
    )
    .orderBy(col("total_revenue").desc())
)

In [0]:
# Sellers are a key entity in marketplace businesses (Flipkart, Amazon, Meesho)
# Seller scorecards are a classic data engineering output

df_gold_sellers = (
    df_silver
    .filter(col("order_status") == "DELIVERED")
    .groupBy("seller_id")
    .agg(
        spark_round(spark_sum("line_total"),    2).alias("total_gmv"),
        spark_round(avg("price"),               2).alias("avg_selling_price"),
        countDistinct("order_id")               .alias("total_orders"),
        countDistinct("customer_unique_id")     .alias("unique_customers_served"),
        countDistinct("product_category_name")  .alias("category_count"),
        spark_round(avg("actual_delivery_days"),2).alias("avg_delivery_days"),
        spark_round(
            spark_sum(when(col("is_late_delivery") == True, 1).otherwise(0))
            / count("*") * 100, 2
        ).alias("late_delivery_pct")
    )
    # Seller tier
    .withColumn("seller_tier",
        when(col("total_gmv") >= 50000, lit("platinum"))
        .when(col("total_gmv") >= 10000, lit("gold"))
        .when(col("total_gmv") >= 1000,  lit("silver"))
        .otherwise(lit("bronze")))
)

# Rank sellers by GMV using window function
w_seller = Window.orderBy(col("total_gmv").desc())
df_gold_sellers = df_gold_sellers \
    .withColumn("gmv_rank", col("total_gmv").cast("double")) \
    .withColumn("seller_rank_overall",
                col("total_gmv").cast("double"))

# Simpler rank approach for Gold output
df_gold_sellers = df_gold_sellers.orderBy(col("total_gmv").desc())

print(f"Seller scorecard rows : {df_gold_sellers.count():,}")
display(df_gold_sellers.limit(15))

In [0]:
from pyspark.sql.functions import lag, coalesce, lit
from pyspark.sql.window import Window

# Month-over-month revenue growth per category
# This requires a window function on the already-aggregated Gold data

w_mom = Window \
    .partitionBy("product_category_name") \
    .orderBy("order_year", "order_month")

df_gold_category_trend = (
    df_gold_revenue
    .select(
        "order_year", "order_month", "order_month_str",
        "product_category_name", "total_revenue",
        "total_orders", "unique_customers"
    )
    # Previous month revenue for this category
    .withColumn("prev_month_revenue",
                lag("total_revenue", 1).over(w_mom))
    # MoM growth %
    .withColumn("mom_revenue_growth_pct",
                spark_round(
                    (col("total_revenue") - col("prev_month_revenue"))
                    / col("prev_month_revenue") * 100,
                    2
                ))
    # Previous month orders
    .withColumn("prev_month_orders",
                lag("total_orders", 1).over(w_mom))
    .withColumn("mom_order_growth_pct",
                spark_round(
                    (col("total_orders") - col("prev_month_orders"))
                    / col("prev_month_orders") * 100,
                    2
                ))
    .filter(col("prev_month_revenue").isNotNull())  # drop first month (no lag)
    .orderBy("product_category_name", "order_year", "order_month")
)

print(f"Category trend rows : {df_gold_category_trend.count():,}")
display(
    df_gold_category_trend
    .orderBy(col("mom_revenue_growth_pct").desc())
    .limit(20)
)

In [0]:
gold_tables = {
    "revenue_by_category_monthly" : (df_gold_revenue,         "order_year"),
    "customer_lifetime_value"     : (df_gold_clv,             "customer_state"),
    "seller_performance"          : (df_gold_sellers,         None),
    "category_mom_trend"          : (df_gold_category_trend,  "order_year"),
}

for table_name, (df, partition_col) in gold_tables.items():
    full_table = f"`{CATALOG}`.`{GOLD_SCHEMA}`.`{table_name}`"
    writer = (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
    )
    if partition_col:
        writer = writer.partitionBy(partition_col)

    writer.saveAsTable(full_table)
    cnt = spark.sql(f"SELECT COUNT(*) as c FROM {full_table}").collect()[0]["c"]
    print(f"  ✓ {full_table}  →  {cnt:,} rows")

print("\nAll Gold tables written.")

In [0]:
# Run OPTIMIZE on the largest Gold tables
tables_to_optimize = [
    f"`{CATALOG}`.`{GOLD_SCHEMA}`.`revenue_by_category_monthly`",
    f"`{CATALOG}`.`{GOLD_SCHEMA}`.`customer_lifetime_value`",
    f"`{CATALOG}`.`{GOLD_SCHEMA}`.`category_mom_trend`",
]

for table in tables_to_optimize:
    spark.sql(f"OPTIMIZE {table}")
    print(f"  ✓ OPTIMIZE done : {table}")

print("\nOptimize complete.")

In [0]:
# VACUUM removes old Parquet files no longer referenced by the Delta log
# Default retention = 7 days (protects time travel)
# WARNING: never run VACUUM with retention < 7 days in production

for table in tables_to_optimize:
    spark.sql(f"VACUUM {table} RETAIN 168 HOURS")  # 168 hours = 7 days
    print(f"  ✓ VACUUM done : {table}")

print("\nVACUUM complete.")

# View full table history
spark.sql(f"""
    DESCRIBE HISTORY `{CATALOG}`.`{GOLD_SCHEMA}`.`revenue_by_category_monthly`
""").select("version", "timestamp", "operation").show(10, truncate=False)

In [0]:
print("=" * 55)
print("TOP 10 CATEGORIES BY TOTAL REVENUE")
print("=" * 55)
display(spark.sql(f"""
    SELECT
        product_category_name,
        SUM(total_revenue)    AS revenue,
        SUM(total_orders)     AS orders,
        ROUND(AVG(avg_order_value), 2) AS avg_order_value,
        ROUND(AVG(late_delivery_rate), 2) AS avg_late_pct
    FROM `{CATALOG}`.`{GOLD_SCHEMA}`.`revenue_by_category_monthly`
    GROUP BY product_category_name
    ORDER BY revenue DESC
    LIMIT 10
"""))

In [0]:
print("=" * 55)
print("CUSTOMER SEGMENT SUMMARY")
print("=" * 55)
display(spark.sql(f"""
    SELECT
        customer_segment,
        COUNT(*)                            AS customers,
        ROUND(AVG(lifetime_revenue), 2)     AS avg_clv,
        ROUND(AVG(total_orders), 2)         AS avg_orders,
        ROUND(SUM(lifetime_revenue), 2)     AS total_revenue_contribution
    FROM `{CATALOG}`.`{GOLD_SCHEMA}`.`customer_lifetime_value`
    GROUP BY customer_segment
    ORDER BY total_revenue_contribution DESC
"""))

In [0]:
print("=" * 55)
print("MONTHLY REVENUE TREND (ALL CATEGORIES COMBINED)")
print("=" * 55)
display(spark.sql(f"""
    SELECT
        order_month_str,
        ROUND(SUM(total_revenue), 2)    AS monthly_revenue,
        SUM(total_orders)               AS monthly_orders,
        SUM(unique_customers)           AS monthly_customers,
        ROUND(AVG(late_delivery_rate),2) AS avg_late_delivery_pct
    FROM `{CATALOG}`.`{GOLD_SCHEMA}`.`revenue_by_category_monthly`
    GROUP BY order_month_str
    ORDER BY order_month_str
"""))